# Detección de ROI facial y normalización de pose

Segunda etapa del preprocesamiento. Toma los clips de 5–7 s que produce
`01_preparar_dataset_equinos.ipynb` y genera frames **recortados a la cara del equino y alineados**,
de modo que el ojo y el hocico caigan siempre en la misma posición.

Cubre el **objetivo específico 2** del plan de trabajo: detección de la región de interés y
alineación de los rostros a lo largo de los clips.

## Por qué hace falta

1. **El fondo predice la clase.** Los 3 videos de Disgustado son interior de box; Interesado y
   Sereno son casi todos exterior. Un modelo entrenado sobre el frame completo puede separar
   Disgustado sin mirar la cara ni una vez.
2. **La pose de la cabeza varía sin control**, de frontal completo a perfil total. Comparar
   microexpresiones entre poses tan distintas no tiene sentido geométrico.

## Requisitos previos

Haber ejecutado `01_preparar_dataset_equinos.ipynb` hasta el Bloque 2, que deja
`inventario_clips.csv` y la carpeta `clips_equinos/`.

**Usar entorno con GPU** (Entorno de ejecución → Cambiar tipo de entorno → GPU).

## Estructura

| Bloque | Pasos | Qué hace |
|---|---|---|
| 1 | 1–4 | Configuración, funciones geométricas e inventario |
| 2 | 5–8 | Detección del equino y suavizado temporal |
| 3 | 9–11 | Keypoints faciales → `keypoints_faciales.csv` |
| 4 | 12–13 | ROI, alineación y recorte |
| 5 | 14–17 | Control de calidad y decisión |

## Paso 0 — Arranque: montar Drive y cargar el paquete `equinos`

In [ ]:
# ============================================================
# ARRANQUE — persistencia en Google Drive
# ============================================================
# Esta celda es autocontenida a propósito: corre ANTES de que el paquete `equinos` esté
# disponible, así que no puede importarlo.
#
# Resuelve en cascada de dónde sale el código:
#   1. de la copia en Drive (RAIZ/codigo/equinos)
#   2. clonando el repo, si hay un GITHUB_TOKEN en los secrets de Colab
#   3. si no, falla con instrucciones concretas
#
# El repo es privado, por eso un `git clone` sin credenciales no alcanza. El token va en los
# secrets de Colab (panel izquierdo, icono de llave) y NUNCA dentro del notebook: lo que se
# escribe en una celda queda en el archivo y se comparte con él.

import shutil
import subprocess
import sys
from pathlib import Path

NOMBRE_PROYECTO = "tesina_equinos"
REPO = "github.com/sergioyanez/Tesina-UGR.git"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

    CODIGO = Path("/content/drive/MyDrive") / NOMBRE_PROYECTO / "codigo"
    CODIGO.mkdir(parents=True, exist_ok=True)

    if not (CODIGO / "equinos").is_dir():
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception as error:
            # Se distingue "el secret no existe" de "existe pero este notebook no tiene acceso".
            # El interruptor por notebook es lo que más se olvida, y sin distinguirlo el mensaje
            # manda a crear un secret que ya está creado.
            if type(error).__name__ == "NotebookAccessError":
                raise SystemExit(
                    "El secret GITHUB_TOKEN existe pero este notebook no tiene acceso.\n"
                    "  Panel izquierdo -> icono de llave -> activá el interruptor de este notebook."
                )
            token = None

        if not token:
            raise SystemExit(
                f"Falta el paquete `equinos` en {CODIGO}.\n"
                "  Opción A: cargá un secret GITHUB_TOKEN en Colab (panel izquierdo, icono de llave).\n"
                f"  Opción B: copiá a mano la carpeta `equinos/` del repo dentro de {CODIGO}."
            )

        clon = Path("/content/_repo_tesina")
        shutil.rmtree(clon, ignore_errors=True)
        resultado = subprocess.run(
            ["git", "clone", "--depth", "1", f"https://{token}@{REPO}", str(clon)],
            capture_output=True, text=True,
        )
        if resultado.returncode != 0:
            # No se imprime el comando ni stderr: contienen el token.
            raise SystemExit("Falló el clonado. Verificá que GITHUB_TOKEN tenga permiso de lectura del repo.")

        shutil.copytree(clon / "equinos", CODIGO / "equinos", dirs_exist_ok=True)
        print(f"Paquete `equinos` copiado a {CODIGO}")

    sys.path.insert(0, str(CODIGO))
else:
    sys.path.insert(0, str(Path.cwd()))

# Si `equinos` ya se importó en este kernel, Python NO vuelve a leer los archivos: se queda con
# lo que tiene en sys.modules. Una sincronización recién hecha quedaría invisible hasta reiniciar
# el entorno, y el síntoma es desconcertante — los archivos nuevos están en Drive y el import
# igual falla. Se purga el caché de módulos para que el import de abajo lea el código vigente.
for _modulo in [m for m in list(sys.modules) if m == "equinos" or m.startswith("equinos.")]:
    del sys.modules[_modulo]

from equinos import cache, checkpoint, entorno, etapas, reporte, rutas

RAIZ = rutas.inicializar()


## Paso 0a — Parámetros de la etapa

In [ ]:
# Parámetros del preprocesamiento facial.
#
# Papermill inyecta una celda justo debajo de ésta cuando el notebook se ejecuta desde
# 00_pipeline.ipynb. Ejecutado a mano, valen estos valores por defecto.
# Están todos juntos acá para poder barrerlos sin tocar el resto del notebook.

CONF_MINIMA_DETECCION = 0.50   # confianza mínima para aceptar una caja de equino
CONF_MINIMA_KEYPOINT  = 0.50   # confianza mínima por keypoint facial
VENTANA_SUAVIZADO     = 5      # frames de la mediana móvil sobre las cajas
MARGEN_ROI            = 0.45   # aire alrededor de la cara, como fracción del lado
ANCHO_RECORTE         = 224    # entrada estándar de los backbones preentrenados

# ENCUADRE — campo de visión del recorte, en unidades de la distancia ojo-hocico `d`.
#
# Tres números independientes, y esa independencia ES el parámetro. Hasta el 28/08/2026 el
# encuadre se declaraba con FRAC_OJO y FRAC_HOCICO sobre un lado único y cuadrado, lo que ataba
# el campo vertical al lateral: no se podía pedir más aire arriba sin pedir más a los costados.
#
# El déficit medido es asimétrico, así que esa atadura escondía el problema por construcción:
#
#   punta de oreja, por encima del ojo:  p50 0,81 d   p75 0,99 d   p90 1,23 d
#   el encuadre viejo mostraba:               0,33 d
#   semiancho de las orejas:             p50 0,57 d   p90 1,11 d
#   el encuadre viejo mostraba:               0,83 d
#
# Falta el cuádruple de aire vertical; lateralmente sobra. Con el encuadre viejo el vector
# base->punta de la oreja —lo que permite leer si está echada hacia atrás, el indicador
# principal de disgusto en equinos— quedaba legible en el 4,3 % de los frames.
#
# Los valores de abajo salen del barrido de `equinos.encuadre`, no de una elección a ojo:
# llevan ese 4,3 % al 77,2 % empatando exacto el relleno mediano del encuadre anterior, porque
# gastan todo el presupuesto en vertical y recortan campo lateral. Ver el change
# `encuadre-que-incluye-las-orejas` y su `medicion.md`.
#
# Para barrerlos sin escribir una sola imagen:
#     from equinos import encuadre
#     encuadre.barrido({"actual": encuadre.CampoDeVision(1.00, 0.30, 0.75)})
AIRE_SOBRE_OJO        = 1.00   # aire por encima del ojo, en unidades de d
AIRE_BAJO_HOCICO      = 0.30   # aire por debajo del hocico, en unidades de d
SEMIANCHO_LATERAL     = 0.75   # semiancho lateral, en unidades de d

# El ALTO del recorte NO se declara: se deriva del aspecto que pide el campo de visión, con el
# ancho fijo. Elegir los dos por separado deformaría la cara o cambiaría el campo sin querer.   # posición vertical canónica del hocico

# Keypoint del hocico. None reproduce el comportamiento histórico —el primer candidato que
# el modelo exponga—, que NO es un criterio: es el orden de una lista escrita a mano. Se fija
# con el resultado de `pose.barrido()`, que compara los candidatos sobre la pose ya inferida.
# El valor `"*"` pide el más confiable de cada frame; antes de usarlo, mirar la dispersión
# de la distancia ojo-hocico que informa el barrido.
BODYPART_HOCICO       = None

# Relleno temporal de keypoints. Un frame sin detección cuyos vecinos SÍ la tienen se estima
# interpolando entre ellos, en vez de caer al recorte de caja. El modelo consume el clip entero,
# así que un solo frame sin alinear contamina la muestra completa.
#
# Los dos topes NO son criterio: salen de la validación de dejar-uno-afuera del change
# `interpolacion-temporal-de-keypoints`, medida el 28/08/2026 sobre 3900 frames detectados.
# Un largo de hueco se acepta si su error mediano y su p90 dentro del recorte no superan los de
# la LÍNEA DE BASE —el desplazamiento entre dos detecciones propias consecutivas, 16,28 px y
# 78,22 px—. Sólo el largo 1 quedó por debajo:
#
#   largo 1 -> mediana 13,99 px   (por debajo de la base: se acepta)
#   largo 2 -> mediana 16,33 px   (ya la supera: no se acepta)
#
# Subirlos sin volver a correr `equinos.interpolacion.informe_validacion()` es inventar posiciones
# que nadie midió. Ponerlos en 0 desactiva el relleno y reproduce el comportamiento anterior.
MAX_HUECO_INTERPOLADO = 1   # frames consecutivos sin detección que se interpolan entre vecinos
MAX_EXTREMO_SOSTENIDO = 0   # frames del borde del clip que sostienen el valor del vecino

# Umbral por debajo del cual el pipeline automático se considera no utilizable
COBERTURA_MINIMA_ACEPTABLE = 0.70

# True descarta lo ya hecho y re-ejecuta la etapa desde cero.
LIMPIAR_ETAPA = False

# True cuando el notebook lo ejecuta 00_pipeline.ipynb. Apaga las barras de progreso: bajo
# papermill cada actualización de tqdm se guarda como una salida más del notebook, y ese
# notebook vive entero en memoria y se reserializa una y otra vez. Con 92 clips eso consume
# toda la RAM de la sesión y termina colgando Colab.
EJECUCION_AUTOMATICA = False


## Bloque 1: Configuración

Paso 1 — Instalar dependencias

Paso 2 — Constantes

Paso 3 — Funciones geométricas

Paso 4 — Cargar inventario de clips

## Paso 1 — Instalar dependencias


In [ ]:
# ============================================================
# DEPENDENCIAS
# ============================================================
# En Colab TODAS las instalaciones van acá, antes de importar nada del stack científico, y en UN
# SOLO comando para que pip resuelva un conjunto consistente en vez de pisarse a sí mismo.
#
# El problema concreto: DeepLabCut fija numpy<2, y Colab viene con scipy y pandas compilados
# contra numpy 2. Instalarlo a secas baja numpy a 1.x y rompe scipy con
# "No module named 'numpy.strings'". Por eso scipy va pinneado a la última serie que convive
# con numpy 1.x.
#
# Y por eso la instalación va en la PRIMERA celda: si numpy ya está importado en memoria cuando
# pip lo reemplaza en disco, todo revienta con "numpy.dtype size changed", y bajo papermill no
# hay forma de reiniciar el kernel para recuperarse.
#
# Fuera de Colab no se instala nada: el entorno ya está construido y pinneado en
# `requirements-local.txt`. Instalar acá rompería el intérprete de la máquina de trabajo, que no
# es descartable como la sesión de Colab. Se verifica en su lugar, incluida la GPU: descubrir
# que se está corriendo en CPU por el tiempo transcurrido es caro.

if EN_COLAB:
    !pip install -q deeplabcut "scipy<1.14" ultralytics opencv-python pandas numpy tqdm matplotlib

entorno.verificar("vision", gpu=True)


## Paso 2 — Rutas de la etapa

In [ ]:
# ============================================================
# RUTAS
# ============================================================
# Todas cuelgan de la raíz de proyecto persistente, que resolvió el Paso 0. Los frames
# alineados van al área de trabajo local y se persisten empaquetados (Paso 13): son ~1472
# archivos chicos, y Drive montado tiene una latencia por archivo que domina el tiempo total.

resultados_dir = rutas.carpeta("resultados")
clips_dir      = rutas.carpeta("clips")
alineados_dir  = rutas.dir_trabajo("frames_alineados")

inventario_clips_path = resultados_dir / "inventario_clips.csv"

if LIMPIAR_ETAPA:
    etapas.limpiar("roi_normalizacion")

print("Clips de entrada:", clips_dir)
print("Frames alineados:", alineados_dir)

def progreso(iterable, total, etiqueta):
    """tqdm cuando hay alguien mirando; una línea cada 10 unidades cuando ejecuta el pipeline."""
    if not EJECUCION_AUTOMATICA:
        from tqdm import tqdm
        yield from tqdm(iterable, total=total, desc=etiqueta)
        return

    import time as _time
    arranque = _time.time()
    for i, elemento in enumerate(iterable, 1):
        yield elemento
        if i % 10 == 0 or i == total:
            print(f"  {etiqueta}: {i}/{total} · {(_time.time() - arranque) / 60:.1f} min", flush=True)


## Paso 3 — Funciones geométricas

Estas funciones son **puras**: no dependen de ningún modelo ni de GPU, y están probadas por
separado con casos sintéticos, incluidos los degenerados. Son la parte del pipeline donde se
esconden los errores silenciosos, así que conviene tenerlas aisladas y verificadas.

In [ ]:
import numpy as np

# ============================================================
# SUAVIZADO TEMPORAL DE CAJAS
# ============================================================

def suavizar_cajas(cajas, ventana=5):
    """
    Aplica una mediana movil a una secuencia de cajas [x1, y1, x2, y2].

    El caballo y la camara se mueven despacio, asi que la caja real varia
    de forma suave entre frames. La mediana elimina los saltos bruscos de
    una deteccion aislada mal ubicada sin correr los bordes, cosa que un
    promedio si haria.

    Las entradas None (frames sin deteccion) se ignoran para calcular la
    mediana, pero se rellenan con el valor suavizado de su posicion.
    """
    n = len(cajas)
    if n == 0:
        return []

    salida = []
    mitad = max(1, ventana // 2)

    for i in range(n):
        desde = max(0, i - mitad)
        hasta = min(n, i + mitad + 1)
        vecinas = [c for c in cajas[desde:hasta] if c is not None]

        if not vecinas:
            salida.append(None)
            continue

        salida.append(tuple(np.median(np.array(vecinas, dtype=float), axis=0)))

    return salida


def rellenar_faltantes(cajas):
    """
    Rellena las posiciones sin deteccion con la caja valida mas cercana.

    Devuelve (cajas_rellenadas, cantidad_rellenada). Si no hay ninguna
    deteccion valida devuelve la lista original y -1.
    """
    validas = [i for i, c in enumerate(cajas) if c is not None]

    if not validas:
        return list(cajas), -1

    salida = list(cajas)
    rellenadas = 0

    for i in range(len(cajas)):
        if cajas[i] is not None:
            continue
        j = min(validas, key=lambda k: abs(k - i))
        salida[i] = cajas[j]
        rellenadas += 1

    return salida, rellenadas


# ============================================================
# TRANSFORMADA DE SIMILITUD A PARTIR DE DOS PUNTOS
# ============================================================

def similitud_desde_dos_puntos(p1, p2, q1, q2):
    """
    Construye la matriz afin 2x3 de la similitud que lleva p1->q1 y p2->q2.

    Dos correspondencias determinan exactamente una similitud (4 grados de
    libertad: escala, rotacion, y traslacion en x e y). Es la misma idea que
    se usa para alinear rostros humanos por la linea entre los ojos; aca la
    referencia es el eje ojo-hocico, que define la orientacion de la cabeza
    tanto de frente como de perfil.

    Devuelve None si los puntos de origen coinciden (no hay eje que alinear).
    """
    p1 = np.asarray(p1, dtype=float)
    p2 = np.asarray(p2, dtype=float)
    q1 = np.asarray(q1, dtype=float)
    q2 = np.asarray(q2, dtype=float)

    v = p2 - p1
    w = q2 - q1

    norma_v = np.hypot(*v)
    if norma_v < 1e-8:
        return None

    escala = np.hypot(*w) / norma_v
    angulo = np.arctan2(w[1], w[0]) - np.arctan2(v[1], v[0])

    cos_a = np.cos(angulo) * escala
    sin_a = np.sin(angulo) * escala

    rotacion = np.array([[cos_a, -sin_a],
                         [sin_a,  cos_a]])

    traslacion = q1 - rotacion @ p1

    return np.array([[rotacion[0, 0], rotacion[0, 1], traslacion[0]],
                     [rotacion[1, 0], rotacion[1, 1], traslacion[1]]])


# ============================================================
# REGION DE INTERES A PARTIR DE KEYPOINTS
# ============================================================

def roi_desde_keypoints(puntos, margen=0.45, forma=None, aspecto=1.0):
    """
    Caja que cubre los keypoints faciales con un margen relativo.

    'aspecto' es alto/ancho. Con 1.0 sale cuadrada, que es el comportamiento
    historico; con el aspecto del recorte canonico sale con la misma forma que
    el camino alineado, para que el dataset no mezcle dos geometrias.

    Que la caja tenga el aspecto del destino es lo que evita que el reescalado
    posterior deforme la cara: lo que deforma no es que la caja sea rectangular,
    es que su aspecto no coincida con el del destino.

    'margen' es la fraccion del lado que se agrega a cada costado.

    Si se pasa 'forma' (alto, ancho), la caja se recorta al borde de la imagen.
    Ojo: recortar cambia el aspecto, y ahi si hay deformacion residual. Es el
    mismo compromiso que tenia la version cuadrada.
    """
    pts = np.asarray([p for p in puntos if p is not None], dtype=float)
    if len(pts) == 0:
        return None

    x1, y1 = pts.min(axis=0)
    x2, y2 = pts.max(axis=0)

    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    ancho = max(x2 - x1, (y2 - y1) / max(aspecto, 1e-9)) * (1 + 2 * margen)
    ancho = max(ancho, 1.0)
    alto = ancho * aspecto

    caja = [cx - ancho / 2, cy - alto / 2, cx + ancho / 2, cy + alto / 2]

    if forma is not None:
        alto, ancho = forma[:2]
        caja[0] = max(0.0, caja[0])
        caja[1] = max(0.0, caja[1])
        caja[2] = min(float(ancho), caja[2])
        caja[3] = min(float(alto), caja[3])

    return tuple(caja)


## Paso 4 — Cargar el inventario de clips

In [ ]:
import pandas as pd

if not inventario_clips_path.exists():
    raise SystemExit(
        f"No existe {inventario_clips_path}.\n"
        "Hay que ejecutar antes 01_preparar_dataset_equinos.ipynb, que lo produce.\n"
        "Desde 00_pipeline.ipynb esto se encadena solo."
    )

df_clips = pd.read_csv(inventario_clips_path)

# Las rutas del CSV son relativas a la raíz de proyecto: se resuelven contra la raíz vigente,
# que puede ser otra sesión de Colab o la máquina local.
df_clips["existe"] = df_clips["ruta_clip"].apply(lambda r: rutas.resolver(r).exists())
faltantes = df_clips[~df_clips["existe"]]

print("Clips en el inventario:", len(df_clips))
print("Clips faltantes en disco:", len(faltantes))

if len(faltantes) > 0:
    print("\nEl inventario referencia clips que no están en la raíz de proyecto.")
    print(faltantes[["clase", "clip"]].head(10))

df_clips = df_clips[df_clips["existe"]].reset_index(drop=True)

print("\nClips por clase:")
print(df_clips["clase"].value_counts())
print("\nCaballos distintos:", df_clips["id_caballo"].nunique())
df_clips.head()


## Bloque 2: Detección del equino

Paso 5 — Cargar el detector y localizar la clase `horse`

Paso 6 — Detectar en cada frame

Paso 7 — Suavizado temporal y relleno de huecos

Paso 8 — Cobertura de la detección

Se usa un detector preentrenado en COCO, que ya incluye la clase `horse` entre sus 80 categorías.
**No requiere ninguna anotación manual.** La caja de cuerpo completo no es el objetivo final, pero
acota el espacio de búsqueda y sirve de respaldo cuando fallan los keypoints faciales.

## Paso 5 — Cargar el detector y localizar la clase `horse`

In [ ]:
from ultralytics import YOLO

# ============================================================
# DETECTOR PREENTRENADO EN COCO
# ============================================================

detector = YOLO("yolo11x.pt")   # se descarga automáticamente la primera vez

# El índice de la clase NO se hardcodea: se busca por nombre en el propio
# modelo. Si en algún momento se cambia de detector, esto sigue funcionando.
nombres = detector.names
CLASE_EQUINO = None

for indice, nombre in nombres.items():
    if nombre.lower() in ("horse", "caballo"):
        CLASE_EQUINO = indice
        break

if CLASE_EQUINO is None:
    raise RuntimeError(
        f"El modelo no expone una clase de equino. Clases disponibles: {list(nombres.values())}"
    )

print(f"Clase de equino encontrada: índice {CLASE_EQUINO} -> '{nombres[CLASE_EQUINO]}'")
print("Total de clases del modelo:", len(nombres))

## Paso 6 — Detectar el equino en cada frame

Se procesan los mismos 16 frames por clip que usa el Bloque 3, para que ambos datasets —el crudo y
el alineado— sean comparables frame a frame.

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

NUM_FRAMES = 16

def indices_uniformes(total_frames, cantidad):
    """Mismos índices que usa el Bloque 3, para poder comparar ambos datasets."""
    if total_frames <= 0:
        return []
    return np.linspace(0, total_frames - 1, cantidad).astype(int).tolist()


detecciones = []   # una fila por frame

for _, fila in progreso(df_clips.iterrows(), len(df_clips), "Detectando"):
    ruta_clip = rutas.resolver(fila["ruta_clip"])
    cap = cv2.VideoCapture(str(ruta_clip))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = indices_uniformes(total, NUM_FRAMES)

    for orden, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()

        registro = {
            "clase": fila["clase"],
            "id_caballo": fila["id_caballo"],
            "video_origen": fila["video_origen"],
            "clip": fila["clip"],
            "ruta_clip": fila["ruta_clip"],
            "orden": orden,
            "frame_idx": int(idx),
            "caja": None,
            "confianza": 0.0,
            "n_detecciones": 0,
        }

        if not ok:
            detecciones.append(registro)
            continue

        resultado = detector.predict(
            frame, classes=[CLASE_EQUINO], conf=CONF_MINIMA_DETECCION, verbose=False
        )[0]

        cajas = resultado.boxes
        registro["n_detecciones"] = 0 if cajas is None else len(cajas)

        if cajas is not None and len(cajas) > 0:
            # Cuando hay más de un equino, se conserva el de mayor confianza
            confianzas = cajas.conf.cpu().numpy()
            mejor = int(np.argmax(confianzas))
            registro["caja"] = tuple(cajas.xyxy.cpu().numpy()[mejor].tolist())
            registro["confianza"] = float(confianzas[mejor])

        detecciones.append(registro)

    cap.release()

df_det = pd.DataFrame(detecciones)

print("Frames procesados:", len(df_det))
print("Frames con detección:", int(df_det["caja"].notna().sum()))
print("Frames con más de un equino:", int((df_det["n_detecciones"] > 1).sum()))
df_det.head()

## Paso 7 — Suavizado temporal y relleno de huecos

La mediana móvil se aplica **por clip**, nunca cruzando clips distintos: dos clips consecutivos del
mismo video pueden tener al animal en posiciones muy diferentes.

In [ ]:
# ============================================================
# SUAVIZADO POR CLIP
# ============================================================

cajas_suaves = {}
relleno_por_clip = {}
clips_sin_deteccion = []

for clip, grupo in df_det.groupby("clip", sort=False):
    grupo = grupo.sort_values("orden")
    cajas = [c if isinstance(c, tuple) else None for c in grupo["caja"]]

    suaves = suavizar_cajas(cajas, ventana=VENTANA_SUAVIZADO)
    rellenas, n_rellenadas = rellenar_faltantes(suaves)

    if n_rellenadas == -1:
        clips_sin_deteccion.append(clip)
        continue

    relleno_por_clip[clip] = n_rellenadas
    for orden, caja in zip(grupo["orden"], rellenas):
        cajas_suaves[(clip, orden)] = caja

df_det["caja_suave"] = [
    cajas_suaves.get((c, o)) for c, o in zip(df_det["clip"], df_det["orden"])
]
df_det["caja_rellenada"] = df_det["caja"].isna() & df_det["caja_suave"].notna()

print("Clips sin ninguna detección (excluidos):", len(clips_sin_deteccion))
for c in clips_sin_deteccion:
    print("   ", c)

print("\nFrames rellenados:", int(df_det["caja_rellenada"].sum()))
print("Frames con caja utilizable:", int(df_det["caja_suave"].notna().sum()))

## Paso 8 — Cobertura de la detección

In [ ]:
# ============================================================
# COBERTURA POR CLASE Y POR CABALLO
# ============================================================

df_det["con_caja"] = df_det["caja_suave"].notna()

print("Cobertura global de detección: {:.1%}".format(df_det["con_caja"].mean()))

print("\nPor clase:")
print(df_det.groupby("clase")["con_caja"].agg(["mean", "count"]).rename(
    columns={"mean": "cobertura", "count": "frames"}))

print("\nPor caballo:")
cob_caballo = df_det.groupby("id_caballo")["con_caja"].mean().sort_values()
print(cob_caballo)

bajos = cob_caballo[cob_caballo < COBERTURA_MINIMA_ACEPTABLE]
if len(bajos) > 0:
    print("\nADVERTENCIA: caballos por debajo del umbral aceptable "
          f"({COBERTURA_MINIMA_ACEPTABLE:.0%}):")
    print(bajos)
else:
    print(f"\nTodos los caballos superan el {COBERTURA_MINIMA_ACEPTABLE:.0%} de cobertura.")

## Bloque 3: Keypoints faciales

Paso 9 — Cargar el modelo de pose

Paso 10 — Descubrir el esquema, seleccionar el par de referencia y extraer los keypoints

Paso 11 — Cobertura de keypoints

Se usa **DeepLabCut SuperAnimal-Quadruped**: estima keypoints en cuadrúpedos sin entrenar ni anotar
nada. Verificado en Colab sobre un clip real (~35 s de inferencia por clip en GPU, más ~10 s de
video etiquetado que después se descarta).

El bloque produce **dos** artefactos, y la separación entre ellos es deliberada.

### Contrato de `pose_completa.csv` — lo caro

Formato largo: una fila por frame y por bodypart. Es la salida cruda del modelo, y se persiste
porque la inferencia es la operación más cara del pipeline mientras que decidir *qué* keypoints
usar es una decisión analítica que se revisa varias veces.

| Columna | Significado |
|---|---|
| `clip` | Nombre del clip |
| `orden` | Índice del frame dentro del clip (0 a 15) |
| `bodypart` | Nombre del keypoint según el esquema del modelo |
| `x`, `y` | Coordenadas en píxeles de la imagen original |
| `confianza` | Likelihood que devuelve el modelo |

Es largo y no ancho porque el conjunto de bodyparts lo fija el modelo, no el proyecto: en formato
ancho, cambiar de modelo cambiaría el esquema de columnas y rompería todo lo que lea el archivo.

### Contrato de `keypoints_faciales.csv` — lo derivado

| Columna | Significado |
|---|---|
| `clip` | Nombre del clip |
| `orden` | Índice del frame dentro del clip (0 a 15) |
| `x_ojo`, `y_ojo` | Coordenadas del ojo en píxeles de la imagen original |
| `conf_ojo` | Confianza del keypoint ocular |
| `x_hocico`, `y_hocico` | Coordenadas del hocico en píxeles de la imagen original |
| `conf_hocico` | Confianza del keypoint del hocico |
| `bodypart_ojo`, `bodypart_hocico` | Qué keypoint se usó en ese frame, para poder auditarlo |

Se deriva de `pose_completa.csv` con `equinos.pose.seleccionar_par()`, así que rehacerlo con otro
keypoint cuesta segundos y no una hora.

Todo lo posterior consume **únicamente** `keypoints_faciales.csv`. Cualquier fuente que lo
produzca sirve: el modelo automático, otro modelo, o anotación manual.

## Paso 9 — Cargar el modelo de pose

Se usa **DeepLabCut SuperAnimal-Quadruped**, que estima keypoints en cuadrúpedos sin entrenar ni
anotar nada (Mathis et al., 2018, citado en la bibliografía del plan de trabajo).

El esquema de keypoints **no se adivina desde un archivo de configuración**: se lee del propio
resultado de la inferencia, que es la única fuente confiable. Por eso este paso sólo carga la
librería y define la función de inferencia; el esquema aparece en el Paso 10.

In [ ]:
# ============================================================
# MODELO DE POSE PARA CUADRÚPEDOS
# ============================================================
# DeepLabCut ya quedó instalado en el Paso 1, junto al resto de las dependencias.

import contextlib
import io

import deeplabcut
import pandas as pd
import numpy as np
from pathlib import Path

print("DeepLabCut:", getattr(deeplabcut, "__version__", "?"))

# Archivos auxiliares que genera DeepLabCut junto a cada clip. Se borran
# después de extraer los keypoints: el .h5 y el .pickle son intermedios y el
# video etiquetado pesa casi como el clip original. Sin limpiarlos, se duplica
# el espacio ocupado y se ensucia clips_equinos/, que otros pasos recorren.
SUFIJOS_AUXILIARES = (".h5", ".pickle", "_labeled_before_adapt.mp4", "_labeled.mp4")


def inferir_keypoints_clip(ruta_clip):
    """
    Corre la inferencia sobre un clip.

    Devuelve el DataFrame de poses (filas = frames, columnas MultiIndex
    scorer/individuals/bodyparts/coords) o None si la inferencia falla.
    """
    ruta_clip = str(ruta_clip)

    # DeepLabCut imprime varias decenas de líneas por video. Bajo papermill esa salida se
    # acumula en el notebook, que vive entero en memoria: con 92 clips es una de las cosas
    # que agotan la RAM de la sesión. Los errores siguen viéndose, sólo se descarta el stdout.
    silenciador = contextlib.redirect_stdout(io.StringIO()) if EJECUCION_AUTOMATICA \
        else contextlib.nullcontext()

    try:
        with silenciador:
            resultado = deeplabcut.video_inference_superanimal(
                [ruta_clip],
                superanimal_name="superanimal_quadruped",
                model_name="hrnet_w32",
                detector_name="fasterrcnn_resnet50_fpn_v2",
            )
    except Exception as e:
        print(f"  fallo en {Path(ruta_clip).name}: {type(e).__name__}: {e}")
        return None

    # El resultado es un dict indexado por la ruta del video. Se toma el único
    # valor en lugar de reconstruir la clave, que puede venir normalizada.
    if not isinstance(resultado, dict) or not resultado:
        print(f"  resultado inesperado en {Path(ruta_clip).name}: {type(resultado)}")
        return None

    return next(iter(resultado.values()))


def limpiar_auxiliares(ruta_clip):
    """Borra los archivos que DeepLabCut deja junto al clip."""
    carpeta = Path(ruta_clip).parent
    raiz = Path(ruta_clip).stem
    borrados = 0
    for archivo in carpeta.glob(f"{raiz}_*"):
        if archivo.suffix in (".h5", ".pickle") or archivo.name.endswith(".mp4"):
            if archivo.name != Path(ruta_clip).name:
                archivo.unlink()
                borrados += 1
    return borrados


print("Función de inferencia lista.")

### Descubrir el esquema de keypoints

Se corre la inferencia sobre **un solo clip** y se lee la lista de keypoints directamente de las
columnas del DataFrame. Es la fuente autoritativa: cualquier otra vía depende de rutas internas de
la librería que cambian entre versiones.

In [ ]:
# ============================================================
# ESQUEMA REAL DEL MODELO
# ============================================================

clip_prueba = rutas.resolver(df_clips.iloc[0]["ruta_clip"])
print("Inferencia de prueba sobre:", Path(clip_prueba).name)

pose_prueba = inferir_keypoints_clip(clip_prueba)

if pose_prueba is None:
    raise RuntimeError("La inferencia de prueba falló. Revisar la instalación de DeepLabCut.")

print("\nFilas (frames):", len(pose_prueba))
print("Niveles de columnas:", pose_prueba.columns.names)

SCORER      = pose_prueba.columns.get_level_values("scorer")[0]
INDIVIDUOS  = list(dict.fromkeys(pose_prueba.columns.get_level_values("individuals")))
BODYPARTS   = list(dict.fromkeys(pose_prueba.columns.get_level_values("bodyparts")))
COORDS      = list(dict.fromkeys(pose_prueba.columns.get_level_values("coords")))

print("Scorer:", SCORER)
print("Individuos:", INDIVIDUOS)
print("Coords:", COORDS)
print(f"\nKeypoints disponibles ({len(BODYPARTS)}):")
for i, b in enumerate(BODYPARTS):
    print(f"  {i:3d}  {b}")

## Paso 10 — Seleccionar el par de referencia y extraer los keypoints

### Por qué se elige el ojo por frame y no uno fijo

En perfil cerrado **sólo se ve un ojo**, y cuál es depende de hacia dónde mire el animal. Fijar
`left_eye` de antemano perdería todos los frames en los que el caballo mira para el otro lado.

Por eso, en cada frame, se toma el ojo con mayor `likelihood`. Es la diferencia entre aprovechar
todos los frames o la mitad.

### Por qué el hocico es un parámetro y no una constante

La primera versión resolvía el hocico recorriendo una lista de candidatos y quedándose con el
primero que existiera, que resultó ser `nose`. El diagnóstico del 18/08/2026 mostró el costo de
ese criterio: con el par ojo/`nose`, **el hocico frena 359 frames y el ojo sólo 10**, y la
cobertura de alineación queda en 59,5 %, debajo del umbral de 70 %.

El problema no era `nose` sino elegir por posición en una lista. Ahora el bodypart del hocico es
un parámetro de la etapa, y cuál conviene se decide con `pose.barrido()` sobre la pose ya
inferida — sin volver a pagar la hora de inferencia.

In [ ]:
# ============================================================
# SELECCIÓN DEL PAR OJO / HOCICO
# ============================================================
# Dos puntos determinan exactamente una similitud (escala, rotación y
# traslación), así que no hace falta ajuste por mínimos cuadrados.
#
# El criterio vive en `equinos.pose`, no acá: es el mismo código que usa el barrido, y ésa es la
# razón por la que el barrido predice la cobertura real en vez de una aproximación.

from equinos import pose as sel_pose

BODYPARTS_OJO = sel_pose.bodyparts_oculares(BODYPARTS)
CANDIDATOS_PRESENTES = sel_pose.candidatos_presentes(BODYPARTS)

print("Keypoints oculares (se elige el más confiable en cada frame):", BODYPARTS_OJO)
print("Candidatos a hocico presentes en el modelo:", CANDIDATOS_PRESENTES)
print("Candidatos ausentes:",
      [c for c in sel_pose.CANDIDATOS_HOCICO
       if c not in {b.lower() for b in CANDIDATOS_PRESENTES}])

if BODYPART_HOCICO is None:
    print("\nBODYPART_HOCICO no fijado -> se usará el primer candidato presente:",
          CANDIDATOS_PRESENTES[0] if CANDIDATOS_PRESENTES else "ninguno")
    print("Correr `pose.barrido()` después de esta etapa para elegirlo con evidencia.")
else:
    print("\nBODYPART_HOCICO fijado por parámetro:", BODYPART_HOCICO)

if not BODYPARTS_OJO or not CANDIDATOS_PRESENTES:
    raise RuntimeError(
        "No se pudo seleccionar el par de referencia. "
        f"Elegir a mano de esta lista: {BODYPARTS}"
    )

### Extracción a formato canónico

La inferencia emite la **pose completa** —todos los bodyparts de cada frame muestreado— y de ahí
se deriva `keypoints_faciales.csv`. Esa separación es lo que vuelve barato revisar qué keypoints
usar: la parte cara (inferir) se paga una vez, y la parte analítica (elegir) se rehace en segundos.

Todo lo que sigue en el notebook consume **únicamente** `keypoints_faciales.csv`. Si en algún
momento hay que reemplazar el modelo de pose por anotación manual, este es el único punto a tocar.

In [ ]:
# ============================================================
# EXTRACCIÓN DE LA POSE A FORMATO LARGO
# ============================================================

def serie_keypoint(pose, bodypart, individuo):
    """Devuelve (x, y, likelihood) como arrays, para un keypoint y un individuo."""
    sub = pose.xs((SCORER, individuo, bodypart), axis=1,
                  level=("scorer", "individuals", "bodyparts"))
    return (sub["x"].to_numpy(dtype=float),
            sub["y"].to_numpy(dtype=float),
            sub["likelihood"].to_numpy(dtype=float))


def elegir_individuo(pose):
    """Con varios animales detectados, se queda con el de mayor confianza media."""
    if len(INDIVIDUOS) == 1:
        return INDIVIDUOS[0]
    mejor, mejor_conf = INDIVIDUOS[0], -1.0
    for ind in INDIVIDUOS:
        confs = [serie_keypoint(pose, b, ind)[2] for b in BODYPARTS]
        media = float(np.nanmean(np.concatenate(confs))) if confs else -1.0
        if media > mejor_conf:
            mejor, mejor_conf = ind, media
    return mejor


def extraer_pose_larga(pose, ordenes_e_indices):
    """
    Traduce el DataFrame de poses a filas (clip, orden, bodypart, x, y, confianza).

    Se emiten TODOS los bodyparts, no sólo el par facial: la inferencia es la operación cara del
    pipeline y descartar el resto obligaba a repetirla cada vez que se revisaba qué keypoints
    usar. Sólo se conservan los frames muestreados, no los del video entero, que es lo que
    mantiene la tabla en un tamaño manejable.
    """
    individuo = elegir_individuo(pose)
    n_filas = len(pose)
    series = {b: serie_keypoint(pose, b, individuo) for b in BODYPARTS}

    filas = []
    for orden, frame_idx in ordenes_e_indices:
        # CAP_PROP_FRAME_COUNT puede sobreestimar; se acota al rango real
        idx = int(min(max(frame_idx, 0), n_filas - 1))
        for bodypart, (xs, ys, cs) in series.items():
            filas.append({
                "orden": int(orden),
                "bodypart": bodypart,
                "x": float(xs[idx]),
                "y": float(ys[idx]),
                "confianza": float(cs[idx]),
            })
    return filas


# --- Verificación sobre el clip de prueba antes de lanzar todos ---
grupo_prueba = df_det[df_det["clip"] == df_clips.iloc[0]["clip"]].sort_values("orden")
muestra = extraer_pose_larga(pose_prueba,
                             list(zip(grupo_prueba["orden"], grupo_prueba["frame_idx"])))
for f in muestra:
    f["clip"] = df_clips.iloc[0]["clip"]

print(f"Filas de pose del clip de prueba: {len(muestra)} "
      f"({len(BODYPARTS)} bodyparts x {len(grupo_prueba)} frames)")

par_prueba = sel_pose.seleccionar_par(muestra, BODYPART_HOCICO)
print("\nPar facial derivado, primeros 3 frames:")
for f in par_prueba[:3]:
    print("  ", f)

conf_media = np.mean([f["conf_ojo"] for f in par_prueba])
print(f"\nConfianza media del ojo en el clip de prueba: {conf_media:.3f}")

In [ ]:
# ============================================================
# INFERENCIA SOBRE TODOS LOS CLIPS  (reanudable)
# ============================================================
# Aproximadamente 45 s por clip en GPU (detector + pose + video etiquetado), así que los clips
# llevan alrededor de una hora: tiempo de sobra para que Colab corte la sesión en el medio.
# Con checkpoint por clip, una interrupción cuesta menos de un minuto de trabajo.
#
# El clip se copia al disco local antes de inferir porque DeepLabCut escribe sus auxiliares
# (.h5, .pickle y un video etiquetado que pesa como el original) junto al archivo de entrada.
# Dejarlos caer en Drive sería lento y llenaría la cuenta de basura temporal.
#
# Lo que el checkpoint acumula es la POSE COMPLETA, no el par facial ya elegido. Es a propósito:
# el par se deriva después, y así cambiar de keypoint no obliga a reinferir. El costo es que el
# CSV parcial crece por el número de bodyparts del modelo.

import shutil

from tqdm import tqdm

# Esta etapa no depende de ningún umbral nuestro: DeepLabCut devuelve coordenadas y su
# likelihood, y el filtrado por CONF_MINIMA_KEYPOINT ocurre después, en el Paso 11. BODYPART_HOCICO
# tampoco entra: al persistirse la pose entera, cambiarlo no invalida la inferencia — sólo obliga a
# re-derivar el par, que cuesta segundos. Por eso la lista de claves relevantes va vacía.
CLAVES_KEYPOINTS = []

cp = checkpoint.abrir("roi_normalizacion", globals(), CLAVES_KEYPOINTS)

# Un checkpoint anterior a este cambio acumuló el par facial (x_ojo, x_hocico...) en vez de la
# pose larga. Reanudar sobre él mezclaría dos esquemas en el mismo CSV y el resultado saldría
# corrupto sin avisar, que es exactamente el modo de fallo que el checkpoint existe para evitar.
_previas = cp.filas(nrows=1)
if len(_previas) and "bodypart" not in _previas.columns:
    print("El checkpoint previo guarda el par facial y no la pose completa: se descarta.")
    cp.limpiar()

tmp_dir = rutas.dir_trabajo("clips_inferencia")
pendientes = cp.pendientes(df_clips["clip"].tolist())
print(f"Clips pendientes: {len(pendientes)} de {len(df_clips)}\n")

clips_fallidos = []
por_procesar = df_clips[df_clips["clip"].isin(pendientes)]

for _, fila in progreso(por_procesar.iterrows(), len(por_procesar), "Keypoints"):
    local = tmp_dir / fila["clip"]
    shutil.copy(rutas.resolver(fila["ruta_clip"]), local)

    try:
        pose = inferir_keypoints_clip(local)

        if pose is None:
            clips_fallidos.append(fila["clip"])
            cp.marcar(fila["clip"])
            continue

        grupo = df_det[df_det["clip"] == fila["clip"]].sort_values("orden")
        ordenes_e_indices = list(zip(grupo["orden"], grupo["frame_idx"]))

        filas_clip = extraer_pose_larga(pose, ordenes_e_indices)
        for f in filas_clip:
            f["clip"] = fila["clip"]

        # Primero las filas, después la marca: si el proceso muere en el medio, lo peor que
        # pasa es reprocesar un clip.
        cp.marcar(fila["clip"], filas_clip)
    finally:
        limpiar_auxiliares(local)
        local.unlink(missing_ok=True)

df_pose = cp.filas()

# --- Artefacto 1: la pose cruda, que es lo caro de producir ---
pose_csv_path = resultados_dir / "pose_completa.csv"
df_pose[list(sel_pose.COLUMNAS_POSE)].to_csv(pose_csv_path, index=False)

print("\nPose completa guardada en:", pose_csv_path)
print(f"Filas: {len(df_pose)}  ({df_pose['bodypart'].nunique()} bodyparts)")
print(f"Tamaño: {pose_csv_path.stat().st_size / 1e6:.1f} MB")

# --- Artefacto 2: el par facial, derivado y barato de rehacer ---
df_kp = pd.DataFrame(sel_pose.seleccionar_par(df_pose.to_dict("records"), BODYPART_HOCICO))

kp_csv_path = resultados_dir / "keypoints_faciales.csv"
df_kp.to_csv(kp_csv_path, index=False)

print("\nKeypoints guardados en:", kp_csv_path)
print("Filas:", len(df_kp))
print("Bodypart del hocico usado:", sorted(df_kp["bodypart_hocico"].unique()))
print("Clips fallidos:", len(clips_fallidos), clips_fallidos)
df_kp.head()

## Paso 11 — Cobertura de keypoints

In [ ]:
# ============================================================
# ¿CUÁNTOS FRAMES TIENEN EL PAR OJO/HOCICO UTILIZABLE?
# ============================================================

df_kp["par_valido"] = (
    (df_kp["conf_ojo"] >= CONF_MINIMA_KEYPOINT) &
    (df_kp["conf_hocico"] >= CONF_MINIMA_KEYPOINT) &
    df_kp["x_ojo"].notna() & df_kp["x_hocico"].notna()
)

df_det = df_det.merge(df_kp, on=["clip", "orden"], how="left")
df_det["par_valido"] = df_det["par_valido"].fillna(False)

print("Cobertura global de keypoints: {:.1%}".format(df_det["par_valido"].mean()))

print("\nPor clase:")
print(df_det.groupby("clase")["par_valido"].agg(["mean", "count"]).rename(
    columns={"mean": "cobertura", "count": "frames"}))

print("\nPor caballo:")
cob_kp = df_det.groupby("id_caballo")["par_valido"].mean().sort_values()
print(cob_kp)

if df_det["par_valido"].mean() < COBERTURA_MINIMA_ACEPTABLE:
    print("\n" + "=" * 60)
    print("COBERTURA INSUFICIENTE")
    print("=" * 60)
    print("El pipeline automático de keypoints no alcanza el umbral aceptable.")
    print("Opciones, en este orden:")
    print("  1. Correr `pose.barrido()` y elegir el keypoint del hocico con evidencia.")
    print("     No requiere reinferir: la pose completa ya está persistida.")
    print("  2. Revisar cuál de los dos keypoints frena, con `reporte.cuello_de_botella()`.")
    print("  3. Anotar manualmente una ROI, acotada a los ejemplares que fallan.")
    print("")
    print("  NO bajar CONF_MINIMA_KEYPOINT: la confianza del par es bimodal, así que relajar")
    print("  el umbral sólo compra keypoints que el propio modelo da por malos, y alinear")
    print("  sobre un keypoint mal puesto es peor que no alinear.")

# ============================================================
# ANOTACIONES MANUALES
# ============================================================
# Se aplican ANTES del relleno temporal y le ganan a la interpolación: una posición que una
# persona miró es mejor evidencia que una estimada desde los vecinos. Sólo se anotan frames que
# el modelo NO detectó, así que no compiten con las detecciones propias.
#
# El archivo lo produce `equinos.anotacion` (./correr_local.sh anotar). Si no existe, esta celda
# no hace nada y el pipeline se comporta como antes.

from equinos import anotacion

manuales = anotacion.leer_manuales()
claves_manuales = set()

if manuales:
    indice = {(c, o): i for i, (c, o) in enumerate(zip(df_det["clip"], df_det["orden"]))}
    aplicadas = 0
    for (clip_m, orden_m), fila_m in manuales.items():
        i = indice.get((clip_m, int(orden_m)))
        if i is None:
            continue
        df_det.loc[df_det.index[i], ["x_ojo", "y_ojo", "x_hocico", "y_hocico"]] = [
            float(fila_m["x_ojo"]), float(fila_m["y_ojo"]),
            float(fila_m["x_hocico"]), float(fila_m["y_hocico"]),
        ]
        df_det.loc[df_det.index[i], "par_valido"] = True
        claves_manuales.add((clip_m, int(orden_m)))
        aplicadas += 1

    print()
    print(f"Anotaciones manuales aplicadas: {aplicadas} de {len(manuales)} en el archivo")
    if aplicadas < len(manuales):
        print(f"  ({len(manuales) - aplicadas} no corresponden a ningún frame del dataset actual)")
else:
    print()
    print("Sin anotaciones manuales (no existe keypoints_manuales.csv).")


# ============================================================
# RELLENO TEMPORAL DE LOS KEYPOINTS FALTANTES
# ============================================================
# Se usa la MISMA función que valida `equinos.interpolacion`, no una segunda implementación: el
# diseño pide un solo camino de construcción, y dos copias divergen sin avisar.
#
# Acá sólo se estiman los PUNTOS. La transformada se construye después, en la celda de recorte,
# con `similitud_desde_dos_puntos()` igual que para cualquier frame detectado.

from equinos import interpolacion

grupos_kp = {
    clip: g.sort_values("orden").to_dict("records")
    for clip, g in df_det.groupby("clip", sort=False)
}
rescates = interpolacion.rescates_por_clip(
    grupos_kp, MAX_HUECO_INTERPOLADO, MAX_EXTREMO_SOSTENIDO)

por_origen = {}
for datos in rescates.values():
    por_origen[datos["origen"]] = por_origen.get(datos["origen"], 0) + 1

print()
print("Relleno temporal (topes: hueco {}, extremo {}):".format(
    MAX_HUECO_INTERPOLADO, MAX_EXTREMO_SOSTENIDO))
if rescates:
    for origen, cuantos in sorted(por_origen.items()):
        print(f"  {origen:<20} {cuantos:>5} frames rescatados")
    detectados = int(df_det["par_valido"].sum())
    print(f"  cobertura geométrica: {detectados} detectados + {len(rescates)} estimados "
          f"= {(detectados + len(rescates)) / len(df_det):.1%}")
else:
    print("  ninguno (topes en 0, o ningún hueco dentro del tope)")

## Bloque 4: ROI y alineación

Paso 12 — Recortar y alinear

Paso 13 — Generar `dataset_frames_alineados.csv`

## Paso 12 — Recortar y alinear

Cada frame sigue uno de dos caminos:

- **Con par ojo/hocico válido** → transformada de similitud que los lleva a la posición canónica.
  El recorte sale alineado.
- **Sin par válido** → ROI cuadrada derivada de la caja de detección, sin alinear. Queda marcado
  con `alineado = False` para poder filtrarlo o analizarlo aparte más adelante.

Nunca se inventan coordenadas: si no hay dato, se registra la ausencia.

In [ ]:
# ============================================================
# RECORTE Y ALINEACIÓN
# ============================================================

from equinos import encuadre

# El campo de visión declarado en la celda de parámetros, y la geometría de destino que se deriva
# de él. El alto NO se elige: sale del aspecto que el campo exige, con el ancho fijo.
CAMPO = encuadre.CampoDeVision(AIRE_SOBRE_OJO, AIRE_BAJO_HOCICO, SEMIANCHO_LATERAL)
GEO = encuadre.destino(CAMPO, ANCHO_RECORTE)
dest_ojo, dest_hocico = GEO["dest_ojo"], GEO["dest_hocico"]
FORMA_DESTINO = (GEO["ancho"], GEO["alto"])

print(f"Encuadre: {AIRE_SOBRE_OJO:.2f} d sobre el ojo, {AIRE_BAJO_HOCICO:.2f} d bajo el hocico, "
      f"±{SEMIANCHO_LATERAL:.2f} d lateral")
print(f"Recorte : {GEO['ancho']}x{GEO['alto']} px  (aspecto {CAMPO.aspecto:.2f}, "
      f"d = {GEO['px_por_d']:.0f} px)")

registros_alineados = []
cache_frames = {}

for clip, grupo in progreso(df_det.groupby("clip", sort=False), df_det["clip"].nunique(), "Alineando"):
    grupo = grupo.sort_values("orden")
    ruta_clip = rutas.resolver(grupo.iloc[0]["ruta_clip"])

    cap = cv2.VideoCapture(str(ruta_clip))
    salida_dir = alineados_dir / grupo.iloc[0]["clase"] / Path(ruta_clip).stem
    salida_dir.mkdir(parents=True, exist_ok=True)

    for _, fila in grupo.iterrows():
        if fila["caja_suave"] is None:
            continue

        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fila["frame_idx"]))
        ok, frame = cap.read()
        if not ok:
            continue

        alineado = False
        origen = "caja"
        relleno_fraccion = float("nan")

        # Los puntos con los que se va a alinear: propios si se detectaron, estimados si el
        # relleno temporal los rescató. De acá en adelante el camino es UNO SOLO — la misma
        # transformada de similitud— y por eso `origen` se registra aparte: es lo único que
        # después distingue una posición observada de una inferida.
        rescate = rescates.get((clip, int(fila["orden"])))
        if fila["par_valido"]:
            puntos = ((fila["x_ojo"], fila["y_ojo"]), (fila["x_hocico"], fila["y_hocico"]))
            # `manual` y `keypoints` construyen el recorte igual; se distinguen sólo para poder
            # contestar cuántas posiciones midió el modelo y cuántas puso una persona.
            origen_candidato = ("manual" if (clip, int(fila["orden"])) in claves_manuales
                                else "keypoints")
        elif rescate is not None:
            puntos = (rescate["ojo"], rescate["hocico"])
            origen_candidato = rescate["origen"]
        else:
            puntos = None
            origen_candidato = None

        if puntos is not None:
            # Camino principal: alineación por transformada de similitud
            M = similitud_desde_dos_puntos(puntos[0], puntos[1], dest_ojo, dest_hocico)
            if M is not None:
                # BORDER_REPLICATE y no el negro por defecto. Cuando el cuadrado canónico se
                # sale del frame —la cabeza contra un borde, o el animal filmado de cerca— el
                # relleno negro es un artefacto de alto contraste, y su cantidad depende de a qué
                # distancia se filmó cada caballo. O sea que correlaciona con el ejemplar, que a
                # su vez predice la clase: exactamente la clase de atajo que una CNN aprende
                # antes que la expresión. Replicar el borde inventa píxeles, pero inventa píxeles
                # sin filo y sin patrón que valga la pena aprender.
                recorte = cv2.warpAffine(frame, M, FORMA_DESTINO,
                                         borderMode=cv2.BORDER_REPLICATE)
                alineado = True
                origen = origen_candidato

                # Cuánto de este recorte son píxeles inventados. Se registra por frame y no sólo
                # se cuenta en agregado, porque el relleno depende de a qué distancia se filmó,
                # que depende del ejemplar, que predice la clase: es un confounder y hay que
                # poder ablacionarlo después, igual que se hace con `origen_alineacion`.
                alto_frame, ancho_frame = frame.shape[:2]
                relleno_fraccion = encuadre.relleno(
                    puntos[0][0], puntos[0][1], puntos[1][0], puntos[1][1],
                    GEO, ancho_frame, alto_frame)

        if not alineado:
            # Degradación: ROI a partir de la caja, sin alinear. Con el MISMO aspecto que el
            # camino alineado: si saliera cuadrada y se reescalara al destino rectangular, estos
            # frames quedarían estirados y el dataset mezclaría dos geometrías, que es
            # exactamente el tipo de diferencia sistemática que después se aprende como atajo.
            x1, y1, x2, y2 = fila["caja_suave"]
            ideal = roi_desde_keypoints([(x1, y1), (x2, y2)],
                                        margen=0.0, aspecto=CAMPO.aspecto)
            roi = roi_desde_keypoints([(x1, y1), (x2, y2)],
                                      margen=0.0, forma=frame.shape, aspecto=CAMPO.aspecto)
            if roi is None:
                continue
            rx1, ry1, rx2, ry2 = [int(round(v)) for v in roi]
            if rx2 - rx1 < 2 or ry2 - ry1 < 2:
                continue
            # Análogo del relleno para este camino: qué fracción de la ROI pedida se perdió
            # contra el borde del cuadro. Acá no se inventan píxeles, se recorta campo — pero es
            # la misma dependencia de la distancia de filmación, y conviene poder medirla igual.
            area_ideal = (ideal[2] - ideal[0]) * (ideal[3] - ideal[1])
            relleno_fraccion = max(0.0, 1 - ((rx2 - rx1) * (ry2 - ry1)) / max(area_ideal, 1e-9))
            recorte = cv2.resize(frame[ry1:ry2, rx1:rx2], FORMA_DESTINO)

        nombre = f"frame_{int(fila['orden']):03d}.jpg"
        ruta_salida = salida_dir / nombre
        cv2.imwrite(str(ruta_salida), recorte)

        registros_alineados.append({
            "clase": fila["clase"],
            "id_caballo": fila["id_caballo"],
            "video_origen": fila["video_origen"],
            "clip": fila["clip"],
            "frame": nombre,
            "ruta_frame": ruta_salida.relative_to(alineados_dir).as_posix(),
            "orden": int(fila["orden"]),
            "alineado": alineado,
            "origen_alineacion": origen,
            "relleno_fraccion": round(float(relleno_fraccion), 4),
            "caja_rellenada": bool(fila["caja_rellenada"]),
            "confianza_deteccion": float(fila["confianza"]),
        })

    cap.release()

df_alineados = pd.DataFrame(registros_alineados)

print("Frames generados:", len(df_alineados))
print("Alineados (geometría): {} ({:.1%})".format(
    int(df_alineados["alineado"].sum()), df_alineados["alineado"].mean()))
print("\nPor origen — un frame detectado y uno estimado NO son lo mismo:")
for origen, cuantos in df_alineados["origen_alineacion"].value_counts().items():
    print(f"  {origen:<20} {cuantos:>5}  ({cuantos / len(df_alineados):.1%})")

relleno_alineados = df_alineados.loc[df_alineados["alineado"], "relleno_fraccion"]
print("\nRelleno del recorte (sólo frames alineados):")
print(f"  mediana {relleno_alineados.median():.1%} | p90 {relleno_alineados.quantile(0.90):.1%} "
      f"| sin relleno {(relleno_alineados == 0).mean():.1%}")
df_alineados.head()

## Paso 13 — Guardar `dataset_frames_alineados.csv`

In [ ]:
alineados_csv_path = resultados_dir / "dataset_frames_alineados.csv"
df_alineados.to_csv(alineados_csv_path, index=False)

print("CSV guardado en:", alineados_csv_path)
print("\nColumnas:", list(df_alineados.columns))

# id_caballo tiene que sobrevivir hasta acá: es la clave de agrupamiento
# para particionar sin fuga entre conjuntos.
assert "id_caballo" in df_alineados.columns, "se perdió la trazabilidad por caballo"
print("\nCaballos representados:", df_alineados["id_caballo"].nunique())
print("Frames por clase:")
print(df_alineados["clase"].value_counts())

# Los frames alineados están en el disco local de la sesión: sin esto se pierden al cerrarla.
cache.empaquetar(alineados_dir, "frames_alineados")


## Bloque 5: Control de calidad

Paso 14 — Reporte de cobertura consolidado

Paso 15 — Inspección visual por clase

Paso 16 — Verificar que la alineación funciona

Paso 17 — Decisión

## Paso 14 — Reporte de cobertura consolidado

In [ ]:
# ============================================================
# REPORTE FINAL
# ============================================================

print("=" * 60)
print("COBERTURA DEL PREPROCESAMIENTO FACIAL")
print("=" * 60)

frames_esperados = len(df_clips) * NUM_FRAMES
print(f"Frames esperados:  {frames_esperados}")
print(f"Frames generados:  {len(df_alineados)} "
      f"({len(df_alineados)/frames_esperados:.1%})")
print(f"Frames alineados:  {int(df_alineados['alineado'].sum())} "
      f"({df_alineados['alineado'].mean():.1%})")

print("\nPor clase:")
resumen = df_alineados.groupby("clase").agg(
    frames=("alineado", "count"),
    alineados=("alineado", "sum"),
    tasa=("alineado", "mean"),
    caballos=("id_caballo", "nunique"),
)
print(resumen)

print("\nPor caballo:")
por_caballo = df_alineados.groupby("id_caballo").agg(
    frames=("alineado", "count"), tasa=("alineado", "mean")
).sort_values("tasa")
print(por_caballo)

problematicos = por_caballo[por_caballo["tasa"] < COBERTURA_MINIMA_ACEPTABLE]
if len(problematicos) > 0:
    print(f"\nADVERTENCIA: {len(problematicos)} caballos por debajo del "
          f"{COBERTURA_MINIMA_ACEPTABLE:.0%} de alineación.")
    print("No continuar al entrenamiento sin decidir qué hacer con ellos.")

# ============================================================
# PERSISTIR EL REPORTE
# ============================================================
# Este es el número que decide si el pipeline automático es utilizable o hay que anotar las 92
# ROI a mano. Imprimirlo no alcanza: la salida de una celda se pierde con la sesión.

reporte = por_caballo.reset_index()
reporte["clase"] = reporte["id_caballo"].map(
    df_alineados.groupby("id_caballo")["clase"].first()
)
reporte["por_debajo_del_umbral"] = reporte["tasa"] < COBERTURA_MINIMA_ACEPTABLE

reporte_path = resultados_dir / "reporte_cobertura.csv"
reporte.to_csv(reporte_path, index=False)

print("\nReporte de cobertura guardado en:", reporte_path)


## Paso 15 — Inspección visual por clase

In [ ]:
import matplotlib.pyplot as plt

# ============================================================
# GRILLA DE RECORTES ALINEADOS
# ============================================================
# Si la normalización funciona, dentro de cada fila las cabezas deben
# aparecer con la misma orientación y escala.
#
# `ruta_frame` se guarda RELATIVA a alineados_dir (Paso 12), porque es lo que define la
# estructura del zip de la caché. Hay que reanclarla acá: pasársela a cv2.imread tal cual la
# resuelve contra el directorio de trabajo, devuelve None y cvtColor levanta una excepción.
#
# Y esta celda es puro control de calidad visual, al final de una etapa de una hora. Un frame
# faltante tiene que quedar registrado, no tumbar la corrida: por eso se cuenta y se sigue.

N_MUESTRAS = 6
clases = sorted(df_alineados["clase"].unique())

fig, axes = plt.subplots(len(clases), N_MUESTRAS,
                         figsize=(N_MUESTRAS * 2.2, len(clases) * 2.4))
axes = np.atleast_2d(axes)

ilegibles = []

for f, clase in enumerate(clases):
    disponibles = df_alineados[(df_alineados["clase"] == clase) &
                               (df_alineados["alineado"])]
    if len(disponibles) == 0:
        disponibles = df_alineados[df_alineados["clase"] == clase]

    muestra = disponibles.sample(min(N_MUESTRAS, len(disponibles)), random_state=42)

    for c in range(N_MUESTRAS):
        ax = axes[f, c]
        ax.axis("off")
        if c >= len(muestra):
            continue

        ruta = alineados_dir / muestra.iloc[c]["ruta_frame"]
        img = cv2.imread(str(ruta))

        if img is None:
            ilegibles.append(str(ruta))
            ax.text(0.5, 0.5, "sin imagen", ha="center", va="center", fontsize=8)
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        if c == 0:
            ax.set_title(clase, loc="left", fontsize=11)

plt.tight_layout()
plt.show()

if ilegibles:
    print(f"ATENCIÓN: {len(ilegibles)} recortes de la muestra no se pudieron leer.")
    print("El CSV los declara pero el archivo no está o no es legible, así que el zip de la")
    print("caché tampoco los tiene. Revisar antes de usar el dataset:")
    for r in ilegibles[:5]:
        print("   ", r)
else:
    print(f"Los {N_MUESTRAS * len(clases)} recortes de la muestra se leyeron correctamente.")

## Paso 16 — Verificar que la alineación realmente funciona

Comprobación cuantitativa, no sólo visual: si la normalización es correcta, al proyectar los
keypoints originales con la misma transformada deben caer en la posición canónica en **todos** los
frames alineados. Es la propiedad que define el paso, y conviene medirla y no suponerla.

In [ ]:
# ============================================================
# VERIFICACIÓN NUMÉRICA DE LA ALINEACIÓN
# ============================================================

validos = df_det[df_det["par_valido"]]
errores = []

for _, fila in validos.iterrows():
    M = similitud_desde_dos_puntos(
        (fila["x_ojo"], fila["y_ojo"]),
        (fila["x_hocico"], fila["y_hocico"]),
        dest_ojo, dest_hocico,
    )
    if M is None:
        continue
    ojo = M[:, :2] @ np.array([fila["x_ojo"], fila["y_ojo"]]) + M[:, 2]
    hoc = M[:, :2] @ np.array([fila["x_hocico"], fila["y_hocico"]]) + M[:, 2]
    errores.append(max(np.hypot(*(ojo - dest_ojo)), np.hypot(*(hoc - dest_hocico))))

if errores:
    print(f"Frames verificados: {len(errores)}")
    print(f"Error máximo de alineación: {max(errores):.2e} píxeles")
    print(f"Error medio: {np.mean(errores):.2e} píxeles")
    assert max(errores) < 1e-6, "la alineación no lleva los puntos a la posición canónica"
    print("\nOK: en todos los frames el ojo y el hocico caen en la posición canónica.")
else:
    print("No hay frames con par válido para verificar.")

## Paso 17 — Decisión

Con los números del Paso 14 sobre la mesa:

**Si la tasa de alineación supera el umbral aceptable** → el objetivo específico 2 queda cumplido.
Continuar al Bloque 4 de modelado usando `dataset_frames_alineados.csv`, y entrenar también sobre
`dataset_frames.csv` (sin alinear) para poder reportar la comparación.

**Si no lo supera** → antes de seguir, y en este orden:

1. Correr el barrido de candidatos y fijar el keypoint del hocico con evidencia:

   ```python
   from equinos import pose
   pose.barrido()
   ```

   Es lo más barato que hay: opera sobre `pose_completa.csv` y no vuelve a inferir. Fue lo que
   destrabó la corrida del 18/08/2026, donde el hocico frenaba 359 frames contra 10 del ojo.

2. Ver cuál de los dos keypoints limita, con `reporte.cuello_de_botella()`, y revisar los recortes
   del Paso 16 antes de dar por buena cualquier elección.

3. Si aun así no alcanza, anotar manualmente la ROI — **acotada a los ejemplares que fallan**, no a
   todos los clips. **No es un fracaso del enfoque sino su red de contención.**

**Lo que ya está descartado:** bajar `CONF_MINIMA_KEYPOINT`. La confianza del par ojo/hocico es
bimodal (o ~0,7-0,85 o ~0,05-0,25, sin masa intermedia), así que relajar de 0,50 a 0,20 sólo suma
frames cuyos keypoints el propio modelo da por malos. Alinear sobre un keypoint mal puesto mete
distorsión geométrica sistemática, que es peor que un recorte honesto sin normalizar.

Anotar la cobertura obtenida en `openspec/project.md` en cualquiera de los casos: es un resultado
del trabajo, no un detalle de implementación.